# 02 — Explore
Load the processed dataset and run sanity checks and diagnostic plots.

In [ ]:
import sys, yaml
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
sys.path.insert(0, '..')

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-RII11-02-ADCPSN010.yml'))
refdes = config['refdes']
ds = xr.open_dataset(f'../data/{refdes}.nc')
ds

## Dataset overview

In [ ]:
print('Dimensions:', dict(ds.sizes))
print('Time range:', ds.time.values[0], '→', ds.time.values[-1])
print('Depth range:', ds.bin_depths.values.min(), '–', ds.bin_depths.values.max(), 'm')
print('Variables:', list(ds.data_vars))

## Velocity time-depth plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, var, label in zip(
    axes,
    ['eastward_sea_water_velocity', 'northward_sea_water_velocity'],
    ['Eastward velocity (m s⁻¹)', 'Northward velocity (m s⁻¹)'],
):
    p = ax.pcolormesh(
        ds.time.values, ds.bin_depths.values,
        ds[var].values.T,
        cmap='RdBu_r', vmin=-0.5, vmax=0.5,
    )
    plt.colorbar(p, ax=ax, label=label)
    ax.set_ylabel('Depth (m)')
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

fig.suptitle(f'{refdes} — Velocity', fontsize=12)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## CTD scalar variables

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, var, label in zip(
    axes,
    ['sea_water_temperature', 'sea_water_practical_salinity', 'sound_speed'],
    ['Temperature (°C)', 'Salinity (PSU)', 'Sound speed (m s⁻¹)'],
):
    ax.plot(ds.time.values, ds[var].values, lw=0.6)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## QARTOD flag coverage

In [ ]:
for var in ['sea_water_temperature_qartod_results', 'sea_water_practical_salinity_qartod_results',
             'eastward_sea_water_velocity_qartod_results', 'northward_sea_water_velocity_qartod_results']:
    if var not in ds:
        continue
    flags = ds[var].values.ravel()
    total = flags.size
    print(f'\n{var}')
    for flag, meaning in [(1,'pass'),(3,'suspect'),(4,'fail'),(9,'missing')]:
        n = (flags == flag).sum()
        print(f'  {flag} ({meaning:8s}): {n:>8,}  ({100*n/total:.1f}%)')

## Echo intensity — first beam

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
p = ax.pcolormesh(
    ds.time.values, ds.bin_depths.values,
    ds['echo_intensity'].sel(beam=1).values.T,
    cmap='viridis', vmin=0, vmax=255,
)
plt.colorbar(p, ax=ax, label='Echo intensity (counts)')
ax.set_ylabel('Depth (m)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title(f'{refdes} — Echo Intensity Beam 1')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()